In [1]:
import pandas as pd
import numpy as np

plfs = pd.read_csv("cleaned_2023_24.csv")
print(plfs.shape)
plfs.head()

(418159, 22)


,quarter,year_label,sector,state_ut_code,sex,age,general_education_level,technical_education_level,current_attendance_status,status_code,...,workplace_location_code,enterprise_type_code,num_workers_enterprise,job_contract_type,eligible_paid_leave,social_security_benefits,industry_code_cws,occupation_code_cws,cws,sub-sample wise multiplier
0,Q1,2023_24,1,2,1,72,8,1,NaN,94,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,94,246798
1,Q1,2023_24,1,2,2,65,6,1,NaN,93,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93,246798
2,Q1,2023_24,1,2,1,40,8,1,NaN,11,...,NaN,1.0,1.0,NaN,NaN,NaN,1.0,611.0,11,246798
3,Q1,2023_24,1,2,2,35,8,1,NaN,21,...,NaN,1.0,1.0,NaN,NaN,NaN,1.0,611.0,21,246798
4,Q1,2023_24,1,2,1,15,7,1,26.0,91,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,91,246798


In [2]:
WEIGHT_COL = "sub-sample wise multiplier"

# ======================
# CONSTANTS / CODES
# ======================
EMPLOYED_CODES = [11, 12, 21, 31, 41, 51]
UNEMPLOYED_CODES = [81, 82]

OWN_ACCOUNT_CODE = 11
UNPAID_FAMILY_CODE = 21

GIG_ENTERPRISE_CODES = [1, 2, 3, 4, 12, 19]

# Broad gig-adjacent NIC divisions for overall gig identification
GIG_INDUSTRY_DIV_CODES = [47, 49, 53, 56, 82, 95, 96]

# 3 grouped gig-industry buckets
TRANSPORT_DIVS      = [49]
DELIVERY_FOOD_DIVS  = [47, 53, 56]
PERSONAL_OTHER_DIVS = [82, 95, 96]

GIG_OCC_PREFIXES = ["5", "8"]

GIG_CWS_CORE       = [11, 12, 21, 41, 51]
GIG_CWS_VULNERABLE = [61, 62, 98]

# ======================
# EDUCATION
# ======================
GE_NO_SCHOOLING_CODES = [1, 2, 3, 4]
GE_PRIMARY_CODES      = [5, 6]
GE_SECONDARY_CODES    = [7, 8, 10, 11]
GE_TERTIARY_CODES     = [12, 13]

NO_TECH_EDU_CODE          = 1
TECH_DEGREE_CODES         = [2, 3, 4, 5, 6]
TECH_DIP_BELOW_GRAD_CODES = [7, 8, 9, 10, 11]
TECH_DIP_GRADPLUS_CODES   = [12, 13, 14, 15, 16]

# Precarity components
NO_WRITTEN_CONTRACT_CODE = 1
NO_PAID_LEAVE_CODE       = 2
NO_SOCSEC_CODE           = 8

# ======================
# HELPERS
# ======================
def nic_division_from_industry_code(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").astype("Int64")
    out = pd.Series(pd.array([pd.NA] * len(s), dtype="Int64"), index=s.index)

    mask5 = s.notna() & (s >= 10000)
    mask4 = s.notna() & (s >= 1000) & (s < 10000)
    mask2 = s.notna() & (s < 1000)

    out.loc[mask5] = (s.loc[mask5] // 1000).astype("Int64")
    out.loc[mask4] = (s.loc[mask4] // 100).astype("Int64")
    out.loc[mask2] = s.loc[mask2].astype("Int64")
    return out

def first_digit_str(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").astype("Int64")
    return s.astype(str).str.replace("<NA>", "", regex=False).str.strip().str[0]

# ======================
# EMPLOYED SUBSET
# ======================
plfs_emp = plfs[plfs["status_code"].isin(EMPLOYED_CODES)].copy()
print("Employed sample size:", len(plfs_emp))

plfs_emp["state_ut_code"] = pd.to_numeric(plfs_emp["state_ut_code"], errors="coerce").astype("Int64")
plfs_emp = plfs_emp[plfs_emp["state_ut_code"] != 26].copy()

plfs_emp["sector_ru"] = pd.to_numeric(plfs_emp["sector"], errors="coerce").map({1: "rural", 2: "urban"})

# ======================
# CONSTRUCT FLAGS
# ======================
plfs_emp["nic_div"]     = nic_division_from_industry_code(plfs_emp["industry_code"])
plfs_emp["nic_div_cws"] = nic_division_from_industry_code(plfs_emp["industry_code_cws"])

# Gig proxies
plfs_emp["gig_workplace"]              = (pd.to_numeric(plfs_emp["workplace_location_code"], errors="coerce") == 99).astype(int)
plfs_emp["gig_enterprise"]             = pd.to_numeric(plfs_emp["enterprise_type_code"], errors="coerce").isin(GIG_ENTERPRISE_CODES).astype(int)
plfs_emp["gig_num_workers_enterprise"] = pd.to_numeric(plfs_emp["num_workers_enterprise"], errors="coerce").isin([1, 9]).astype(int)

plfs_emp["gig_industry"]     = plfs_emp["nic_div"].isin(GIG_INDUSTRY_DIV_CODES).astype(int)
plfs_emp["gig_industry_cws"] = plfs_emp["nic_div_cws"].isin(GIG_INDUSTRY_DIV_CODES).astype(int)

# 3 grouped industry flags (principal status)
plfs_emp["gig_transport_industry"]      = plfs_emp["nic_div"].isin(TRANSPORT_DIVS).astype(int)
plfs_emp["gig_delivery_food_industry"]  = plfs_emp["nic_div"].isin(DELIVERY_FOOD_DIVS).astype(int)
plfs_emp["gig_personal_other_industry"] = plfs_emp["nic_div"].isin(PERSONAL_OTHER_DIVS).astype(int)

# Optional grouped flags (CWS industry)
plfs_emp["gig_transport_industry_cws"]      = plfs_emp["nic_div_cws"].isin(TRANSPORT_DIVS).astype(int)
plfs_emp["gig_delivery_food_industry_cws"]  = plfs_emp["nic_div_cws"].isin(DELIVERY_FOOD_DIVS).astype(int)
plfs_emp["gig_personal_other_industry_cws"] = plfs_emp["nic_div_cws"].isin(PERSONAL_OTHER_DIVS).astype(int)

plfs_emp["gig_occupation"]     = first_digit_str(plfs_emp["occupation_code"]).isin(GIG_OCC_PREFIXES).astype(int)
plfs_emp["gig_occupation_cws"] = first_digit_str(plfs_emp["occupation_code_cws"]).isin(GIG_OCC_PREFIXES).astype(int)

plfs_emp["gig_cws_core"]   = plfs_emp["cws"].isin(GIG_CWS_CORE).astype(int)
plfs_emp["cws_vulnerable"] = plfs_emp["cws"].isin(GIG_CWS_VULNERABLE).astype(int)

# Inclusive gig worker ID
plfs_emp["gig_worker"] = (
    (
        (plfs_emp["gig_workplace"] == 1) |
        (plfs_emp["gig_enterprise"] == 1) |
        (plfs_emp["gig_num_workers_enterprise"] == 1) |
        (plfs_emp["gig_industry"] == 1) |
        (plfs_emp["gig_occupation"] == 1) |
        (plfs_emp["gig_industry_cws"] == 1) |
        (plfs_emp["gig_occupation_cws"] == 1)
    )
    &
    (
        (plfs_emp["gig_cws_core"] == 1) |
        (plfs_emp["cws_vulnerable"] == 1)
    )
).astype(int)

plfs_emp["non_gig_worker"] = (plfs_emp["gig_worker"] == 0).astype(int)

# Vulnerable employment flags
sc = pd.to_numeric(plfs_emp["status_code"], errors="coerce")
plfs_emp["is_own_account"]    = (sc == OWN_ACCOUNT_CODE).astype(int)
plfs_emp["is_unpaid_family"]  = (sc == UNPAID_FAMILY_CODE).astype(int)
plfs_emp["is_vulnerable_emp"] = (plfs_emp["is_own_account"] | plfs_emp["is_unpaid_family"]).astype(int)

# Age bins
plfs_emp["age_15_24"] = plfs_emp["age"].between(15, 24).astype(int)
plfs_emp["age_25_34"] = plfs_emp["age"].between(25, 34).astype(int)
plfs_emp["age_35_44"] = plfs_emp["age"].between(35, 44).astype(int)
plfs_emp["age_45_54"] = plfs_emp["age"].between(45, 54).astype(int)
plfs_emp["age_55p"]   = (plfs_emp["age"] >= 55).astype(int)

# General education group flags
ge = pd.to_numeric(plfs_emp["general_education_level"], errors="coerce")
plfs_emp["edu_no_schooling"] = ge.isin(GE_NO_SCHOOLING_CODES).astype(int)
plfs_emp["edu_primary"]      = ge.isin(GE_PRIMARY_CODES).astype(int)
plfs_emp["edu_secondary"]    = ge.isin(GE_SECONDARY_CODES).astype(int)
plfs_emp["edu_tertiary"]     = ge.isin(GE_TERTIARY_CODES).astype(int)

# Technical education group flags
te = pd.to_numeric(plfs_emp["technical_education_level"], errors="coerce")
plfs_emp["tech_none"]               = (te == NO_TECH_EDU_CODE).astype(int)
plfs_emp["tech_degree"]             = te.isin(TECH_DEGREE_CODES).astype(int)
plfs_emp["tech_diploma_below_grad"] = te.isin(TECH_DIP_BELOW_GRAD_CODES).astype(int)
plfs_emp["tech_diploma_gradplus"]   = te.isin(TECH_DIP_GRADPLUS_CODES).astype(int)
plfs_emp["tech_any"]                = ((te.notna()) & (te != NO_TECH_EDU_CODE)).astype(int)

# Precarity components
jc = pd.to_numeric(plfs_emp["job_contract_type"], errors="coerce")
plfs_emp["no_written_contract"] = (jc == NO_WRITTEN_CONTRACT_CODE).astype(int)

pl = pd.to_numeric(plfs_emp["eligible_paid_leave"], errors="coerce")
plfs_emp["no_paid_leave"] = (pl == NO_PAID_LEAVE_CODE).astype(int)

ss = pd.to_numeric(plfs_emp["social_security_benefits"], errors="coerce")
plfs_emp["no_social_security"] = (ss == NO_SOCSEC_CODE).astype(int)

# Detailed NIC division flags
for div in GIG_INDUSTRY_DIV_CODES:
    plfs_emp[f"nic_div_{div}"] = (plfs_emp["nic_div"] == div).astype(int)

# ======================
# AGGREGATE TO state × quarter × sector × year_label
# ======================
if "year_label" not in plfs_emp.columns:
    raise ValueError("year_label not found. Ensure your cleaned file contains it.")

group_keys = ["state_ut_code", "quarter", "sector_ru", "year_label"]

agg = (
    plfs_emp
    .groupby(group_keys, observed=False)
    .apply(
        lambda x: pd.Series({
            # denominators
            "employed_weighted":         np.sum(x[WEIGHT_COL]),
            "gig_employed_weighted":     np.sum(x[WEIGHT_COL] * x["gig_worker"]),
            "non_gig_employed_weighted": np.sum(x[WEIGHT_COL] * x["non_gig_worker"]),

            # vulnerable employment numerators
            "vulnerable_emp_weighted":  np.sum(x[WEIGHT_COL] * x["is_vulnerable_emp"]),
            "gig_vulnerable_emp_w":     np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["is_vulnerable_emp"]),
            "non_gig_vulnerable_emp_w": np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["is_vulnerable_emp"]),

            # sex
            "gig_male_weighted":       np.sum(x[WEIGHT_COL] * x["gig_worker"] * (x["sex"] == 1)),
            "gig_female_weighted":     np.sum(x[WEIGHT_COL] * x["gig_worker"] * (x["sex"] == 2)),
            "non_gig_male_weighted":   np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * (x["sex"] == 1)),
            "non_gig_female_weighted": np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * (x["sex"] == 2)),

            # age
            "gig_age_15_24_w": np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["age_15_24"]),
            "gig_age_25_34_w": np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["age_25_34"]),
            "gig_age_35_44_w": np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["age_35_44"]),
            "gig_age_45_54_w": np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["age_45_54"]),
            "gig_age_55p_w":   np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["age_55p"]),

            "non_gig_age_15_24_w": np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["age_15_24"]),
            "non_gig_age_25_34_w": np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["age_25_34"]),
            "non_gig_age_35_44_w": np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["age_35_44"]),
            "non_gig_age_45_54_w": np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["age_45_54"]),
            "non_gig_age_55p_w":   np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["age_55p"]),

            # general education
            "gig_edu_no_schooling_w": np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["edu_no_schooling"]),
            "gig_edu_primary_w":      np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["edu_primary"]),
            "gig_edu_secondary_w":    np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["edu_secondary"]),
            "gig_edu_tertiary_w":     np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["edu_tertiary"]),

            "non_gig_edu_no_schooling_w": np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["edu_no_schooling"]),
            "non_gig_edu_primary_w":      np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["edu_primary"]),
            "non_gig_edu_secondary_w":    np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["edu_secondary"]),
            "non_gig_edu_tertiary_w":     np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["edu_tertiary"]),

            # technical education
            "gig_tech_none_w":           np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["tech_none"]),
            "gig_tech_any_w":            np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["tech_any"]),
            "gig_tech_degree_w":         np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["tech_degree"]),
            "gig_tech_dip_below_grad_w": np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["tech_diploma_below_grad"]),
            "gig_tech_dip_gradplus_w":   np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["tech_diploma_gradplus"]),

            "non_gig_tech_none_w":           np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["tech_none"]),
            "non_gig_tech_any_w":            np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["tech_any"]),
            "non_gig_tech_degree_w":         np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["tech_degree"]),
            "non_gig_tech_dip_below_grad_w": np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["tech_diploma_below_grad"]),
            "non_gig_tech_dip_gradplus_w":   np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["tech_diploma_gradplus"]),

            # exposure proxies
            "gig_industry_weighted":               np.sum(x[WEIGHT_COL] * x["gig_industry"]),
            "gig_occupation_weighted":             np.sum(x[WEIGHT_COL] * x["gig_occupation"]),
            "gig_workplace_weighted":              np.sum(x[WEIGHT_COL] * x["gig_workplace"]),
            "gig_enterprise_weighted":             np.sum(x[WEIGHT_COL] * x["gig_enterprise"]),
            "gig_num_workers_enterprise_weighted": np.sum(x[WEIGHT_COL] * x["gig_num_workers_enterprise"]),

            # CWS-based exposure proxies
            "gig_occupation_cws_weighted": np.sum(x[WEIGHT_COL] * x["gig_occupation_cws"]),
            "gig_industry_cws_weighted":   np.sum(x[WEIGHT_COL] * x["gig_industry_cws"]),

            # weekly vulnerability proxy
            "cws_vulnerable_weighted": np.sum(x[WEIGHT_COL] * x["cws_vulnerable"]),
            "gig_cws_core_weighted":   np.sum(x[WEIGHT_COL] * x["gig_cws_core"]),

            # Z components
            "gig_no_written_contract_w": np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["no_written_contract"]),
            "gig_no_paid_leave_w":       np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["no_paid_leave"]),
            "gig_no_socsec_w":           np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["no_social_security"]),

            "non_gig_no_written_contract_w": np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["no_written_contract"]),
            "non_gig_no_paid_leave_w":       np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["no_paid_leave"]),
            "non_gig_no_socsec_w":           np.sum(x[WEIGHT_COL] * x["non_gig_worker"] * x["no_social_security"]),

            # grouped industry counts within gig
            "gig_transport_w":      np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["gig_transport_industry"]),
            "gig_delivery_food_w":  np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["gig_delivery_food_industry"]),
            "gig_personal_other_w": np.sum(x[WEIGHT_COL] * x["gig_worker"] * x["gig_personal_other_industry"]),

            # detailed concentration within gig
            **{f"gig_div{div}_w": np.sum(x[WEIGHT_COL] * x["gig_worker"] * x[f"nic_div_{div}"]) for div in GIG_INDUSTRY_DIV_CODES},

            "n_employed_unweighted": len(x)
        }),
        include_groups=False
    )
    .reset_index()
)

# ======================
# SHARES
# ======================
den_all    = agg["employed_weighted"].replace({0: np.nan})
den_gig    = agg["gig_employed_weighted"].replace({0: np.nan})
den_nongig = agg["non_gig_employed_weighted"].replace({0: np.nan})

# grouped industry shares within gig
agg["gig_transport_share_within_gig"]      = agg["gig_transport_w"] / den_gig
agg["gig_delivery_food_share_within_gig"]  = agg["gig_delivery_food_w"] / den_gig
agg["gig_personal_other_share_within_gig"] = agg["gig_personal_other_w"] / den_gig

# ======================
# DEMOGRAPHIC SHARES
# ======================
agg["gig_male_share_within_gig"]   = agg["gig_male_weighted"] / den_gig
agg["gig_female_share_within_gig"] = agg["gig_female_weighted"] / den_gig

agg["gig_age_15_24_share_within_gig"] = agg["gig_age_15_24_w"] / den_gig
agg["gig_age_25_34_share_within_gig"] = agg["gig_age_25_34_w"] / den_gig
agg["gig_age_35_44_share_within_gig"] = agg["gig_age_35_44_w"] / den_gig
agg["gig_age_45_54_share_within_gig"] = agg["gig_age_45_54_w"] / den_gig
agg["gig_age_55p_share_within_gig"]   = agg["gig_age_55p_w"] / den_gig

agg["gig_edu_no_schooling_share_within_gig"] = agg["gig_edu_no_schooling_w"] / den_gig
agg["gig_edu_primary_share_within_gig"]      = agg["gig_edu_primary_w"] / den_gig
agg["gig_edu_secondary_share_within_gig"]    = agg["gig_edu_secondary_w"] / den_gig
agg["gig_edu_tertiary_share_within_gig"]     = agg["gig_edu_tertiary_w"] / den_gig

agg["gig_tech_none_share_within_gig"]           = agg["gig_tech_none_w"] / den_gig
agg["gig_tech_any_share_within_gig"]            = agg["gig_tech_any_w"] / den_gig
agg["gig_tech_degree_share_within_gig"]         = agg["gig_tech_degree_w"] / den_gig
agg["gig_tech_dip_below_grad_share_within_gig"] = agg["gig_tech_dip_below_grad_w"] / den_gig
agg["gig_tech_dip_gradplus_share_within_gig"]   = agg["gig_tech_dip_gradplus_w"] / den_gig

agg["non_gig_male_share_within_non_gig"]   = agg["non_gig_male_weighted"] / den_nongig
agg["non_gig_female_share_within_non_gig"] = agg["non_gig_female_weighted"] / den_nongig

agg["non_gig_age_15_24_share_within_non_gig"] = agg["non_gig_age_15_24_w"] / den_nongig
agg["non_gig_age_25_34_share_within_non_gig"] = agg["non_gig_age_25_34_w"] / den_nongig
agg["non_gig_age_35_44_share_within_non_gig"] = agg["non_gig_age_35_44_w"] / den_nongig
agg["non_gig_age_45_54_share_within_non_gig"] = agg["non_gig_age_45_54_w"] / den_nongig
agg["non_gig_age_55p_share_within_non_gig"]   = agg["non_gig_age_55p_w"] / den_nongig

agg["non_gig_edu_no_schooling_share_within_non_gig"] = agg["non_gig_edu_no_schooling_w"] / den_nongig
agg["non_gig_edu_primary_share_within_non_gig"]      = agg["non_gig_edu_primary_w"] / den_nongig
agg["non_gig_edu_secondary_share_within_non_gig"]    = agg["non_gig_edu_secondary_w"] / den_nongig
agg["non_gig_edu_tertiary_share_within_non_gig"]     = agg["non_gig_edu_tertiary_w"] / den_nongig

agg["non_gig_tech_none_share_within_non_gig"]           = agg["non_gig_tech_none_w"] / den_nongig
agg["non_gig_tech_any_share_within_non_gig"]            = agg["non_gig_tech_any_w"] / den_nongig
agg["non_gig_tech_degree_share_within_non_gig"]         = agg["non_gig_tech_degree_w"] / den_nongig
agg["non_gig_tech_dip_below_grad_share_within_non_gig"] = agg["non_gig_tech_dip_below_grad_w"] / den_nongig
agg["non_gig_tech_dip_gradplus_share_within_non_gig"]   = agg["non_gig_tech_dip_gradplus_w"] / den_nongig

# Detailed concentration within gig for key NIC divisions
for div in GIG_INDUSTRY_DIV_CODES:
    agg[f"gig_div{div}_share_within_gig"] = agg[f"gig_div{div}_w"] / den_gig

# ======================
# OUTCOMES / EXPOSURES
# ======================
agg["vulnerable_employment_share"]         = agg["vulnerable_emp_weighted"] / den_all
agg["gig_vulnerable_employment_share"]     = agg["gig_vulnerable_emp_w"] / den_gig
agg["non_gig_vulnerable_employment_share"] = agg["non_gig_vulnerable_emp_w"] / den_nongig

agg["gig_occupation_code_share"]         = agg["gig_occupation_weighted"] / den_all
agg["gig_industry_code_share"]           = agg["gig_industry_weighted"] / den_all
agg["gig_workplace_location_code_share"] = agg["gig_workplace_weighted"] / den_all
agg["gig_enterprise_type_code_share"]    = agg["gig_enterprise_weighted"] / den_all
agg["gig_num_workers_enterprise_share"]  = agg["gig_num_workers_enterprise_weighted"] / den_all

agg["gig_occupation_code_cws_share"] = agg["gig_occupation_cws_weighted"] / den_all
agg["gig_industry_code_cws_share"]   = agg["gig_industry_cws_weighted"] / den_all
agg["gig_volatility_wedge_occ"]      = (agg["gig_occupation_code_cws_share"] - agg["gig_occupation_code_share"]).abs()

agg["cws_vulnerability_rate"] = agg["cws_vulnerable_weighted"] / den_all
agg["gig_cws_core_share"]     = agg["gig_cws_core_weighted"] / den_all

agg["gig_no_written_contract_share"] = agg["gig_no_written_contract_w"] / den_gig
agg["gig_no_paid_leave_share"]       = agg["gig_no_paid_leave_w"] / den_gig
agg["gig_no_socsec_share"]           = agg["gig_no_socsec_w"] / den_gig

agg["non_gig_no_written_contract_share"] = agg["non_gig_no_written_contract_w"] / den_nongig
agg["non_gig_no_paid_leave_share"]       = agg["non_gig_no_paid_leave_w"] / den_nongig
agg["non_gig_no_socsec_share"]           = agg["non_gig_no_socsec_w"] / den_nongig

agg["gig_precarity_index"] = (
    agg["gig_no_written_contract_share"].fillna(0) +
    agg["gig_no_paid_leave_share"].fillna(0) +
    agg["gig_no_socsec_share"].fillna(0)
) / 3

agg["non_gig_precarity_index"] = (
    agg["non_gig_no_written_contract_share"].fillna(0) +
    agg["non_gig_no_paid_leave_share"].fillna(0) +
    agg["non_gig_no_socsec_share"].fillna(0)
) / 3

# ======================
# LABOUR FORCE RATES (15–64)
# ======================
plfs_wa = plfs[plfs["age"].between(15, 64)].copy()
plfs_wa["state_ut_code"] = pd.to_numeric(plfs_wa["state_ut_code"], errors="coerce").astype("Int64")
plfs_wa = plfs_wa[plfs_wa["state_ut_code"] != 26].copy()
plfs_wa["sector_ru"] = pd.to_numeric(plfs_wa["sector"], errors="coerce").map({1: "rural", 2: "urban"})

plfs_wa["is_employed"]   = plfs_wa["status_code"].isin(EMPLOYED_CODES).astype(int)
plfs_wa["is_unemployed"] = plfs_wa["status_code"].isin(UNEMPLOYED_CODES).astype(int)

plfs_lf = plfs_wa[(plfs_wa["is_employed"] == 1) | (plfs_wa["is_unemployed"] == 1)].copy()

lf_rates = (
    plfs_lf
    .groupby(group_keys, observed=False)
    .apply(
        lambda x: pd.Series({
            "labour_force_weighted_15_64":     np.sum(x[WEIGHT_COL]),
            "employed_in_lf_weighted_15_64":   np.sum(x[WEIGHT_COL] * x["is_employed"]),
            "unemployed_in_lf_weighted_15_64": np.sum(x[WEIGHT_COL] * x["is_unemployed"]),
        }),
        include_groups=False
    )
    .reset_index()
)

lf_den = lf_rates["labour_force_weighted_15_64"].replace({0: np.nan})
lf_rates["employment_rate_lf_15_64"] = lf_rates["employed_in_lf_weighted_15_64"] / lf_den
lf_rates["unemployment_rate_15_64"]  = lf_rates["unemployed_in_lf_weighted_15_64"] / lf_den

agg = agg.merge(
    lf_rates[["state_ut_code", "quarter", "sector_ru", "year_label",
              "employment_rate_lf_15_64", "unemployment_rate_15_64"]],
    on=["state_ut_code", "quarter", "sector_ru", "year_label"],
    how="left"
)

# ======================
# FINAL OUTPUT
# ======================
state_mapping = {
    1: 'Jammu & Kashmir', 2: 'Himachal Pradesh', 3: 'Punjab', 4: 'Chandigarh',
    5: 'Uttarakhand', 6: 'Haryana', 7: 'Delhi', 8: 'Rajasthan', 9: 'Uttar Pradesh',
    10: 'Bihar', 11: 'Sikkim', 12: 'Arunachal Pradesh', 13: 'Nagaland', 14: 'Manipur',
    15: 'Mizoram', 16: 'Tripura', 17: 'Meghalaya', 18: 'Assam', 19: 'West Bengal',
    20: 'Jharkhand', 21: 'Odisha', 22: 'Chhattisgarh', 23: 'Madhya Pradesh',
    24: 'Gujarat', 25: 'D & N. Haveli & Daman & Diu', 27: 'Maharashtra',
    28: 'Andhra Pradesh', 29: 'Karnataka', 30: 'Goa', 31: 'Lakshadweep',
    32: 'Kerala', 33: 'Tamilnadu', 34: 'Puduchery', 35: 'Andaman & N. Island',
    36: 'Telangana', 37: 'Ladakh'
}
if "state" not in agg.columns:
    agg["state"] = agg["state_ut_code"].map(state_mapping).fillna(agg["state_ut_code"].astype(str))

if "gig_age_15_24_share_within_gig" in agg.columns and "gig_age_25_34_share_within_gig" in agg.columns:
    agg["gig_age_15_34_share_within_gig"] = (
        agg["gig_age_15_24_share_within_gig"] + agg["gig_age_25_34_share_within_gig"]
    )

if "gig_edu_secondary_share_within_gig" in agg.columns and "gig_edu_tertiary_share_within_gig" in agg.columns:
    agg["gig_edu_secondary_tertiary_share_within_gig"] = (
        agg["gig_edu_secondary_share_within_gig"] + agg["gig_edu_tertiary_share_within_gig"]
    )

agg["gig_employment_share_principal"] = agg["gig_employed_weighted"] / agg["employed_weighted"].replace({0: np.nan})

final_cols = [
    "year_label", "quarter", "state_ut_code", "state", "sector_ru",

    "employed_weighted", "gig_employed_weighted", "non_gig_employed_weighted",

    # outcome Y
    "vulnerable_employment_share",
    "gig_vulnerable_employment_share",
    "non_gig_vulnerable_employment_share",

    # X exposure candidates
    "gig_employment_share_principal",
    "gig_occupation_code_share",
    "gig_industry_code_share",
    "gig_workplace_location_code_share",
    "gig_enterprise_type_code_share",
    "gig_num_workers_enterprise_share",

    # grouped industry composition within gig
    "gig_transport_w",
    "gig_delivery_food_w",
    "gig_personal_other_w",
    "gig_transport_share_within_gig",
    "gig_delivery_food_share_within_gig",
    "gig_personal_other_share_within_gig",

    # volatility
    "gig_occupation_code_cws_share",
    "gig_volatility_wedge_occ",

    # optional
    "cws_vulnerability_rate",

    # Z components + index
    "gig_no_written_contract_share",
    "gig_no_socsec_share",
    "gig_no_paid_leave_share",
    "gig_precarity_index",
    "non_gig_no_written_contract_share",
    "non_gig_no_socsec_share",
    "non_gig_no_paid_leave_share",
    "non_gig_precarity_index",

    # gig demographics
    "gig_male_share_within_gig", "gig_female_share_within_gig",
    "gig_age_15_24_share_within_gig", "gig_age_25_34_share_within_gig",
    "gig_age_35_44_share_within_gig", "gig_age_45_54_share_within_gig",
    "gig_age_55p_share_within_gig",
    "gig_age_15_34_share_within_gig",

    "gig_edu_no_schooling_share_within_gig",
    "gig_edu_primary_share_within_gig",
    "gig_edu_secondary_share_within_gig",
    "gig_edu_tertiary_share_within_gig",
    "gig_edu_secondary_tertiary_share_within_gig",

    "gig_tech_none_share_within_gig",
    "gig_tech_any_share_within_gig",
    "gig_tech_degree_share_within_gig",
    "gig_tech_dip_below_grad_share_within_gig",
    "gig_tech_dip_gradplus_share_within_gig",

    # non-gig demographics
    "non_gig_male_share_within_non_gig", "non_gig_female_share_within_non_gig",
    "non_gig_age_15_24_share_within_non_gig", "non_gig_age_25_34_share_within_non_gig",
    "non_gig_age_35_44_share_within_non_gig", "non_gig_age_45_54_share_within_non_gig",
    "non_gig_age_55p_share_within_non_gig",

    "non_gig_edu_no_schooling_share_within_non_gig",
    "non_gig_edu_primary_share_within_non_gig",
    "non_gig_edu_secondary_share_within_non_gig",
    "non_gig_edu_tertiary_share_within_non_gig",

    "non_gig_tech_none_share_within_non_gig",
    "non_gig_tech_any_share_within_non_gig",
    "non_gig_tech_degree_share_within_non_gig",
    "non_gig_tech_dip_below_grad_share_within_non_gig",
    "non_gig_tech_dip_gradplus_share_within_non_gig",

    # detailed concentration within gig
    "gig_div47_share_within_gig", "gig_div49_share_within_gig",
    "gig_div53_share_within_gig", "gig_div56_share_within_gig",
    "gig_div82_share_within_gig", "gig_div95_share_within_gig",
    "gig_div96_share_within_gig",

    # labour force rates
    "employment_rate_lf_15_64",
    "unemployment_rate_15_64",
]

final_cols = [c for c in final_cols if c in agg.columns]

final_df = agg[final_cols].copy()

quarter_order = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4, "1": 1, "2": 2, "3": 3, "4": 4}
final_df["q_ord"] = final_df["quarter"].map(quarter_order).fillna(9).astype(int)
final_df = (
    final_df.sort_values(["year_label", "q_ord", "state_ut_code", "sector_ru"])
            .drop(columns=["q_ord"])
            .reset_index(drop=True)
)

out_path = "plfs_2023_24.csv"
final_df.to_csv(out_path, index=False)
print("✓ Saved:", out_path)
print(final_df.head(10))

Employed sample size: 164523
✓ Saved: plfs_2023_24.csv
  year_label quarter  state_ut_code             state sector_ru  \
0    2023_24      Q1              1   Jammu & Kashmir     rural   
1    2023_24      Q1              1   Jammu & Kashmir     urban   
2    2023_24      Q1              2  Himachal Pradesh     rural   
3    2023_24      Q1              2  Himachal Pradesh     urban   
4    2023_24      Q1              3            Punjab     rural   
5    2023_24      Q1              3            Punjab     urban   
6    2023_24      Q1              4        Chandigarh     urban   
7    2023_24      Q1              5       Uttarakhand     rural   
8    2023_24      Q1              5       Uttarakhand     urban   
9    2023_24      Q1              6           Haryana     rural   

   employed_weighted  gig_employed_weighted  non_gig_employed_weighted  \
0          642957741              459035584                  183922157   
1          142863773               68964356                